# Diffusion — decay of a magnetic Fourier mode — 3D

`MagneticStandalone3D` · ASN 28 validation case (TRUST-LiMe).

## Introduction

Validation of the **standalone magnetic (induction) equation** added by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data (no Navier–Stokes coupling in this first phase). Every case is
**self-contained** (inline `Mailler`/`Pave` mesh, no `.med` file) and records
point probes (`Postraitement/Sondes`) into `.son` files which are compared
below against a closed-form solution.

This notebook runs **one and only one test case**: ``diffusion``. Per the TRUST
validation-form layout (`Validation/Rapports_automatiques/<Category>/<FormName>/`),
the other `MagneticStandalone3D` cases live in their own sibling folders, each
with `<FormName>/<FormName>.ipynb` + `src/<case>.data`.

## What it validates

$b(x,y,0)=\sin(\pi y)$, Dirichlet in $Y$, zero flux in $X$, $u=0$, $\eta=0.05$:

$$b(x,y,t)=\sin(\pi y)\,e^{-\eta\pi^2t}.$$

We compare the *amplitude ratio* $b(t)/b(0)$ (independent of where the probe is
snapped) to the analytic factor, over all recorded times, and measure the
relaxation rate to make sure the transport coefficient is $\eta$ itself.

Mesh convergence (against `diffusion_refined`) is checked in the
`diffusion_refined` notebook.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| 1 | transport coefficient is $\eta$ itself — **no** $\rho C_p$ leak from the energy base class | measured rate vs $\eta\pi^2$ |
| 2 | $\eta$ used verbatim as diffusion coefficient | decay rate |
| 8 | Dirichlet prescribes $b$; Neumann = zero flux | implied analytic rate |


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    cands = [f"{BUILD}/coarse/{case}_{name}.son",
             f"{BUILD}/{case}_{name}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{name}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def ratio_error(case, tags, ref):
    """Max |b(t)/b(0) - ref(t)| over the given probes and all recorded times."""
    worst = 0.0
    for tag in tags:
        t, b = read_son(case, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - ref(t)))))
    return worst

print("BUILD =", BUILD)


## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).


In [ ]:
from trustutils import run

NB_PROCS = 2

run.reset()
# --- mesh-convergence scan (shared ladder) ----------------------------------
# The single dataset src/diffusion.data carries $nx/$ny node-count placeholders;
# every scale runs the SAME file via trustutils substitution
# (run.addCaseFromTemplate -> string.Template) into build/<scale>/ — one
# data-file name across scales:
#   coarse   = the baseline mesh
#   standard = cells x2 (h/2)
#   fine     = cells x4 (h/4)
# Each scale runs in its own build/<scale>/ directory so that the
# partitioned DOM_000*.Zones of one level never overwrite another's.
NY0 = 65                              # baseline y node count (coarse); x = z = 9
meshes = {
    "coarse":   {"nx": 9, "ny": 65, "nz": 9},            # the baseline mesh
    "standard": {"nx": 9, "ny": 2*NY0-1, "nz": 9},
    "fine":     {"nx": 9, "ny": 4*NY0-3, "nz": 9},
}
cases = {}
for lvl, dic in meshes.items():
    cases[lvl] = run.addCaseFromTemplate("diffusion.data", lvl, dic=dic,
                                         nbProcs=NB_PROCS, excluNR=True)
    cases[lvl].partition()          # trust -partition diffusion.data 2 -> PAR_diffusion.data + DOM_000*.Zones
run.printCases()
run.runCases()
case = cases["coarse"]            # alias kept for the read-back cells


## Checks


In [ ]:
ETA, K = 0.05, np.pi
def analytic_ratio(t): return np.exp(-ETA * K**2 * np.asarray(t, float))

TAGS = ["SONDE_Y025", "SONDE_Y050", "SONDE_Y075"]
err_coarse = ratio_error(case.dataFileName_, TAGS, analytic_ratio)
print(f"coarse  max |ratio - analytic| : {err_coarse:.4e}")
assert err_coarse < 0.05, f"coarse decay error too large: {err_coarse:.3e}"
print("OK: decay matches the analytic solution.")

# --- Business contract: the transport coefficient is eta ITSELF -------
# The class derives from the ENERGY equation, so the real danger is an
# inherited rho*Cp time factor (~1.2e3 for air). Measure the actual
# relaxation rate lambda = -d/dt ln(b/b0) and compare with eta*pi^2.
t, b = read_son(case.dataFileName_, "SONDE_Y050")
lam_measured = -np.log(abs(b[-1] / b[0])) / t[-1]
lam_expected = ETA * np.pi**2
rel = abs(lam_measured - lam_expected) / lam_expected
print(f"measured relaxation rate : {lam_measured:.6f} (expected {lam_expected:.6f})")
print(f"relative error           : {rel:.3e}")
assert rel < 0.05, f"transport coefficient is not eta: rel. error {rel:.3e}"
print("OK: transport coefficient is exactly eta (no rho*Cp leak).")

import matplotlib.pyplot as plt
t, b = read_son(case.dataFileName_, "SONDE_Y050")
plt.figure(figsize=(6,4))
plt.plot(t, b/b[0], "o-", ms=2, label="TRUST-LiMe (coarse)")
plt.plot(t, analytic_ratio(t), "k--", label="analytic exp(-eta pi^2 t)")
plt.xlabel("t [s]"); plt.ylabel("b(t)/b(0)"); plt.legend(); plt.grid(True); plt.show()


## Mesh convergence

The shared refinement ladder, adapted to the 1D eigenmode physics: only the
active `y` direction is refined (`coarse` = `9×65×9`, `standard` = y cells ×2
(`9×129×9`), `fine` = y cells ×4 (`9×257×9`)).

The *Checks* ratio metric saturates at the implicit-Euler temporal error,
common to every mesh. To expose the **spatial** error, the scan compares
$b(t)/b(0)$ against the exact implicit-Euler decay built with the analytic
eigenvalue $\lambda = \eta\pi^2$ on the probe's own time grid,
$\mathrm{ref}(t_n) = \prod_j 1/(1+\lambda\,\Delta t_j)$: the time scheme
cancels out and the residual is the spatial one, $\lambda_h - \lambda =
O(h^2)$. The observed order is $\log_2(e_k/e_{k+1})$.

**Merged form.** The historical `DiffusionRefined` twin is absorbed by
this scan: its role (mesh-independence of the decay) is covered by the
convergence levels asserted above. The twin's exact mesh is the scan's
`standard` in 2D; the 3D twin refined the passive directions as well, which
the y-only ladder covers by design.

In [ ]:
SONDE_TAGS_SCAN = ["SONDE_Y025", "SONDE_Y050", "SONDE_Y075"]
ETA = 0.05                       # magnetic diffusivity of the dataset

def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def implicit_ref(t, lam):
    """Exact implicit-Euler decay ratio on the probe time grid (any dt sequence)."""
    t = np.asarray(t, dtype=float)
    ref = np.ones_like(t)
    for i in range(1, len(t)):
        ref[i:] *= 1.0 / (1.0 + lam * (t[i] - t[i-1]))
    return ref

LAM = ETA * np.pi**2                # analytic eigenvalue eta*pi^2
def spatial_ratio_error(subdir, case_name):
    """Max |b(t)/b(0) - implicit-Euler analytic decay| over the scan probes."""
    worst = 0.0
    for tag in SONDE_TAGS_SCAN:
        t, b = read_son_dir(subdir, case_name, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - implicit_ref(t, LAM)))))
    return worst

scan = {"coarse": ("coarse", case.dataFileName_),
        "standard": (f"standard", cases["standard"].dataFileName_),
        "fine": (f"fine", cases["fine"].dataFileName_)}
scan_nodes = {"coarse": (9, 65, 9), "standard": (9, 129, 9), "fine": (9, 257, 9)}
scan_err = {lvl: spatial_ratio_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes       h_y        |ratio - implicit ref|   observed order")
prev = None
for lvl in ("coarse", "standard", "fine"):
    nx, ny, nz = scan_nodes[lvl]
    e = scan_err[lvl]
    order = "--"
    if prev is not None and prev > 0 and e > 0:
        order = f"{np.log2(prev / e):.2f}"
    print(f"{lvl:<9} {nx:>2}x{ny}x{nz:<2} {1.0/(ny-1):<10.4g} {e:<26.4e} {order}")
    prev = e

assert scan_err["standard"] < scan_err["coarse"], f"no convergence coarse -> standard: {scan_err}"
assert scan_err["fine"] < scan_err["standard"], f"no convergence standard -> fine: {scan_err}"
print("OK: spatial error decreases monotonically under refinement (order ~2).")

## Computer Performance


In [ ]:
run.tablePerf()